# Model 4: Gated Modality-Attention Fusion
**ICT 4442: Deep Learning Course Project**  
Explainable Multimodal ECG Diagnostic Classification Using ECG Signals and Clinical Text

This notebook trains and evaluates **Model 4 (Gated Modality-Attention Network)** combining 12-lead ECG waveforms and leakage-masked clinical text.

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import Dataset, DataLoader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
class ECGEncoder(nn.Module):
    """1D-CNN for raw 12-lead ECG waveforms (B, 12, 5000)."""
    def __init__(self, embed_dim: int = 128):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            nn.Conv1d(12, 32, kernel_size=7, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(5),
            nn.Conv1d(128, 128, kernel_size=7, padding=3),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(5),
        )
        self.adaptive_pool = nn.AdaptiveAvgPool1d(1)
        self.proj = nn.Linear(128, embed_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.conv_blocks(x)
        feat = self.adaptive_pool(feat).squeeze(-1)
        return self.proj(feat)


class TextEncoder(nn.Module):
    """Bi-LSTM for residual clinical text tokens."""
    def __init__(self, vocab_size: int = 202, embed_dim: int = 64, hidden_dim: int = 64, out_dim: int = 128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        self.proj = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        lengths_clamped = torch.clamp(lengths.cpu(), min=1)
        packed = pack_padded_sequence(embedded, lengths_clamped, batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        text_emb = torch.cat((h_n[-2], h_n[-1]), dim=-1)
        return self.proj(text_emb)


class Model4GatedFusion(nn.Module):
    """
    Model 4: Gated Modality-Attention Network
    Computes dynamic scalar softmax weights [a_ecg, a_text] over projected embeddings.
    """
    def __init__(self, vocab_size: int = 202, embed_dim: int = 128, num_classes: int = 5, dropout: float = 0.3):
        super().__init__()
        self.ecg_encoder = ECGEncoder(embed_dim=embed_dim)
        self.text_encoder = TextEncoder(vocab_size=vocab_size, embed_dim=64, hidden_dim=64, out_dim=embed_dim)
        self.gate_net = nn.Linear(embed_dim * 2, 2)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, ecg: torch.Tensor, input_ids: torch.Tensor, lengths: torch.Tensor):
        p_ecg = self.ecg_encoder(ecg)                    # (B, 128)
        p_text = self.text_encoder(input_ids, lengths)   # (B, 128)
        concat_emb = torch.cat([p_ecg, p_text], dim=-1)  # (B, 256)
        gates = torch.softmax(self.gate_net(concat_emb), dim=-1)  # (B, 2) -> [a_ecg, a_text]
        a_ecg = gates[:, 0:1]
        a_text = gates[:, 1:2]
        z_gated = a_ecg * p_ecg + a_text * p_text
        logits = self.classifier(self.dropout(z_gated))
        return logits, gates

In [ ]:
def train_model4(model, train_loader, epochs=20, device=device):
    model = model.to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam([
        {'params': model.ecg_encoder.parameters(), 'lr': 1e-3},
        {'params': model.text_encoder.parameters(), 'lr': 3e-4},
        {'params': model.gate_net.parameters(), 'lr': 1e-3},
        {'params': model.classifier.parameters(), 'lr': 1e-3},
    ])
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for batch in train_loader:
            ecg = batch['ecg'].to(device)
            input_ids = batch['input_ids'].to(device)
            lengths = batch['lengths'].to(device)
            labels = batch['labels'].to(device)
            optimizer.zero_grad()
            logits, _ = model(ecg, input_ids, lengths)
            loss = criterion(logits, labels.float())
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        if epoch % 5 == 0 or epoch == epochs:
            print(f'Epoch {epoch}/{epochs} - Loss: {total_loss / max(1, len(train_loader)):.4f}')
    return model


def evaluate_model4(model, test_loader, output_dir='outputs/model_4_attention_fusion', thresholds=None, device=device):
    model.eval()
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    all_logits, all_labels, all_gates = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            ecg = batch['ecg'].to(device)
            input_ids = batch['input_ids'].to(device)
            lengths = batch['lengths'].to(device)
            labels = batch['labels'].to(device)
            logits, gates = model(ecg, input_ids, lengths)
            all_logits.append(logits.cpu().numpy())
            all_labels.append(labels.cpu().numpy())
            all_gates.append(gates.cpu().numpy())
    probs = 1 / (1 + np.exp(-np.vstack(all_logits)))
    y_true = np.vstack(all_labels)
    gates_arr = np.vstack(all_gates)
    mean_ecg_weight = float(np.mean(gates_arr[:, 0]))
    mean_text_weight = float(np.mean(gates_arr[:, 1]))
    gate_stats = {'mean_ecg_weight': mean_ecg_weight, 'mean_text_weight': mean_text_weight}
    with open(out_path / 'attention_gates.json', 'w') as f:
        json.dump(gate_stats, f, indent=2)
    if thresholds is None:
        thresholds = [0.5] * 5
    y_pred = (probs >= np.array(thresholds)).astype(int)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)
    print(f'Evaluation Results: Macro F1: {macro_f1:.4f}, Micro F1: {micro_f1:.4f}')
    print(f'Gates Logged: ECG={mean_ecg_weight:.3f}, Text={mean_text_weight:.3f}')
    return {'macro_f1': macro_f1, 'micro_f1': micro_f1, 'gates': gate_stats}